# Yalova land parcel valuation
CME434 Geographic Information Systems, Karabük University

SUBHI YASER SUBHI DDA (2310213583) · Magd Shalaby (2210213583)  
Supervisor: Dr. Öğr. Üyesi Sohaib K. M. Abujayyab

Run the geographic section immediately. The valuation section reads the active private training version after the team's manual cohort has been validated and trained.

In [ ]:
from pathlib import Path
import json, os, sys
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if not (root / 'engine.py').is_file():
    root = root.parent
sys.path.insert(0, str(root))
from engine import FEATURES
context = json.loads((root / 'analysis/geographic_context.json').read_text())
pd.DataFrame(context['distance_summary_m']).T

In [ ]:
ax = gpd.read_file(root / 'layers/boundary.geojson').to_crs(32635).plot(color='#d8e4d7', edgecolor='#37594b')
roads = gpd.read_file(root / 'layers/road.geojson').to_crs(32635)
extent = ax.axis()
roads.plot(ax=ax, color='#967860', linewidth=.4)
ax.axis(extent)
ax.set_title('Terrestrial Yalova and mapped main roads')
ax.set_axis_off()

## Selected model and spatial validation
The app runs engine.train after ingestion. Metrics below come exclusively from the active model version. An absent marker indicates that empirical results are pending.

In [ ]:
store = Path(os.environ.get('YALOVA_STATE', root / 'private_state'))
marker = store / 'active.json'
if not marker.exists():
    raise RuntimeError('Complete a real training run before opening the valuation section.')
v = store / 'versions' / json.loads(marker.read_text())['version']
metrics = json.loads((v / 'metrics.json').read_text())
display(pd.DataFrame(metrics['models']), pd.DataFrame(metrics['folds']))
print(metrics['parameters'], metrics['moran'])

In [ ]:
pred = pd.read_csv(v / 'held_out_predictions.csv')
assert pred.groupby('block_5km').outer_fold.nunique().max() == 1
points = pd.read_csv(v / 'regression_dataset.csv')
assert set(FEATURES).issubset(points.columns)
print(points[['unit_tl_m2','area_m2']].describe())
pd.read_csv(v / 'shap_summary.csv').head(12).set_index('feature').iloc[::-1].plot.barh(legend=False, xlabel='Mean absolute SHAP (TL/m²)')

In [ ]:
gpd.read_file(v / 'lisa.geojson').to_crs(32635).plot(column='cluster', categorical=True, legend=True, markersize=10)
plt.title('Local Moran clusters, unadjusted p ≤ 0.05')
plt.axis('off')
plt.show()
grid = gpd.read_file(v / 'grid.geojson').to_crs(32635)
grid.plot(column='price_index', cmap='viridis', legend=True, linewidth=0)
plt.title('Reference parcel index, study-area median = 100')
plt.axis('off')
plt.show()